# Slice-Optimization Benchmark Results

Plots for the latest `results/<run-id>/results.csv` produced by `benchmark.py`. The two knobs under test are **preemptive slice creation** (`preemptive_create_horizon_ms`) and the **slice-recycle pool** (`recycle_pool_size`); the four on/off corners isolate each effect and their interaction. VLDB/SIGMOD-style figures.

Run `pip install pandas matplotlib seaborn` if not already available.

## Reading the config columns

Each row is one benchmarked cell (one `systest -b` invocation, averaged over `runs`). The sweep is the cartesian product of:

- `query` — `SLICE_AGG` (1-day tumbling agg, ~23k live slices) or `SLICE_JOIN` (1-hour join, ~3.6k live slices).
- `preemptive_horizon_ms` — build-side lookahead. **0 = off** (no preemptive creation). With a `preemptive_predictor` it's a wall-clock horizon converted to a slice count; with `off` it's a static event-time lookahead.
- `preemptive_predictor` — watermark predictor driving the wall-clock horizon (`off` = static event-time lookahead).
- `recycle_pool_size` — LIFO of slices retired by probe-side GC, reused build-side. **0 = off**.
- `threads` — worker threads (e.g. 1 vs 16).
- `predictor` — watermark predictor for the predictive *spill* policy (`off` = plain slice store, no spill wrapper). Independent of `preemptive_predictor`.

**The 2×2 corners** (the headline): `preempt = preemptive_horizon_ms > 0`, `recycle = recycle_pool_size > 0`.

- `off/off` — baseline (neither optimization).
- `preempt` — preemptive creation only.
- `recycle` — recycle pool only.
- `both` — both on (interaction).

**Metrics** (higher = better for throughput, lower = better for time): `tuples_per_second`, `bytes_per_second`, `time_s`. Cells with a non-empty `failure_reason` have no metrics and are dropped from the accuracy plots (and tallied separately at the end).

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

# Latest run by default; set RESULTS_CSV to a specific results.csv to override.
RESULTS_CSV = None
if RESULTS_CSV is None:
    candidates = sorted(Path('results').glob('*/results.csv'))
    if not candidates:
        raise FileNotFoundError('no results/<run-id>/results.csv — run benchmark.py first')
    RESULTS_CSV = candidates[-1]
csv_path = Path(RESULTS_CSV)
df = pd.read_csv(csv_path)
print('loaded', csv_path, '|', df.shape)

# Numeric coercion: failed cells leave the metric columns blank -> NaN.
for c in ['preemptive_horizon_ms', 'recycle_pool_size', 'threads', 'runs',
          'time_s', 'tuples_per_second', 'bytes_per_second']:
    df[c] = pd.to_numeric(df[c], errors='coerce')
df['failure_reason'] = df['failure_reason'].fillna('')
df['failed'] = df['failure_reason'].str.len() > 0

# The 2x2 corner: each knob on iff its sentinel-0 is exceeded.
CORNER_ORDER = ['off/off', 'preempt', 'recycle', 'both']
def corner(r):
    p, rec = r['preemptive_horizon_ms'] > 0, r['recycle_pool_size'] > 0
    return {(False, False): 'off/off', (True, False): 'preempt',
            (False, True): 'recycle', (True, True): 'both'}[(p, rec)]
df['corner'] = df.apply(corner, axis=1)
df['corner'] = pd.Categorical(df['corner'], categories=CORNER_ORDER, ordered=True)
df['tps_M'] = df['tuples_per_second'] / 1e6

# ok = scored cells only (drop failures) for the throughput/time plots.
ok = df[~df['failed']].copy()
QUERIES = sorted(df['query'].unique())
THREADS = sorted(df['threads'].dropna().unique().astype(int))
PREDICTORS = sorted(df['predictor'].unique())
print('queries:', QUERIES, '| threads:', THREADS, '| predictors:', PREDICTORS,
      '| failed:', int(df['failed'].sum()), '/', len(df))
ok.head()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# VLDB / SIGMOD camera-ready style: serif, compact, no chartjunk.
sns.set_theme(context='paper', style='whitegrid', font='serif')
plt.rcParams.update({
    'figure.dpi': 130,
    'savefig.dpi': 300,
    'font.size': 11,
    'axes.titlesize': 11,
    'axes.labelsize': 11,
    'axes.titleweight': 'bold',
    'legend.fontsize': 9,
    'legend.frameon': False,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.linewidth': 0.5,
    'grid.alpha': 0.4,
})
CORNER_PALETTE = dict(zip(CORNER_ORDER, sns.color_palette('tab10', 4)))
PRED_PALETTE = dict(zip(PREDICTORS, sns.color_palette('Set2', len(PREDICTORS))))
print('style ready')

## 1. The 2×2 corners — throughput
Mean tuples/s for each on/off corner, one panel per query, bars grouped by thread count. This is the headline: does turning on preemptive creation and/or the recycle pool move throughput, and do they compound? Averages over the predictor settings — the per-predictor breakdown is in §4.

In [ ]:
fig, axes = plt.subplots(1, len(QUERIES), figsize=(4.2 * len(QUERIES), 3.6), squeeze=False)
for ax, q in zip(axes[0], QUERIES):
    sub = ok[ok['query'] == q]
    sns.barplot(sub, x='corner', y='tps_M', hue='threads', order=CORNER_ORDER,
                palette='viridis', ax=ax)
    ax.set_title(q)
    ax.set_xlabel('')
    ax.set_ylabel('throughput (M tuples/s)')
    ax.legend(title='threads', fontsize=8)
fig.suptitle('Throughput by slice-optimization corner (mean over predictors)')
fig.tight_layout()
plt.show()

## 2. Speedup over the off/off baseline
Each corner's throughput divided by the `off/off` baseline at the **same** (query, threads, preemptive_predictor, predictor) cell, so only the two slice knobs vary. >1 = faster than baseline. The dashed line is parity.

In [ ]:
# Join each scored cell to its off/off twin (same everything except the two knobs).
KEY = ['query', 'threads', 'preemptive_predictor', 'predictor']
base = (ok[ok['corner'] == 'off/off']
        .groupby(KEY, as_index=False)['tuples_per_second'].mean()
        .rename(columns={'tuples_per_second': 'base_tps'}))
sp = ok.merge(base, on=KEY, how='left')
sp['speedup'] = sp['tuples_per_second'] / sp['base_tps']
sp = sp[sp['base_tps'].notna()]

fig, axes = plt.subplots(1, len(QUERIES), figsize=(4.2 * len(QUERIES), 3.6), squeeze=False, sharey=True)
for ax, q in zip(axes[0], QUERIES):
    sns.barplot(sp[sp['query'] == q], x='corner', y='speedup', hue='threads',
                order=CORNER_ORDER, palette='viridis', ax=ax)
    ax.axhline(1.0, ls='--', lw=1, color='0.4')
    ax.set_title(q)
    ax.set_xlabel('')
    ax.set_ylabel('speedup vs off/off')
    ax.legend(title='threads', fontsize=8)
fig.suptitle('Speedup over baseline (same query/threads/predictors, only slice knobs vary)')
fig.tight_layout()
plt.show()

## 3. Thread scaling
Throughput vs worker threads, one line per corner, one panel per query. Does either optimization improve how the slice store scales with parallelism?

In [ ]:
fig, axes = plt.subplots(1, len(QUERIES), figsize=(4.2 * len(QUERIES), 3.6), squeeze=False)
for ax, q in zip(axes[0], QUERIES):
    sns.lineplot(ok[ok['query'] == q], x='threads', y='tps_M', hue='corner',
                 hue_order=CORNER_ORDER, palette=CORNER_PALETTE, marker='o', errorbar='sd', ax=ax)
    ax.set_title(q)
    ax.set_xlabel('worker threads')
    ax.set_ylabel('throughput (M tuples/s)')
    ax.set_xticks(THREADS)
    ax.legend(title='corner', fontsize=8)
fig.suptitle('Thread scaling by corner')
fig.tight_layout()
plt.show()

## 4. Predictor breakdown
Throughput per spill `predictor` (`off` = no spill wrapper), grouped by corner, one panel per query at the highest thread count. Shows whether a given predictor's spill policy interacts with the slice knobs.

In [ ]:
tmax = max(THREADS) if THREADS else None
fig, axes = plt.subplots(1, len(QUERIES), figsize=(4.6 * len(QUERIES), 3.6), squeeze=False, sharey=True)
for ax, q in zip(axes[0], QUERIES):
    sub = ok[(ok['query'] == q) & (ok['threads'] == tmax)]
    sns.barplot(sub, x='predictor', y='tps_M', hue='corner', order=PREDICTORS,
                hue_order=CORNER_ORDER, palette=CORNER_PALETTE, ax=ax)
    ax.set_title(f'{q}  (threads={tmax})')
    ax.set_xlabel('spill predictor')
    ax.set_ylabel('throughput (M tuples/s)')
    ax.tick_params(axis='x', rotation=20)
    ax.legend(title='corner', fontsize=8)
fig.suptitle('Throughput by spill predictor and corner')
fig.tight_layout()
plt.show()

## 5. Full-sweep heatmap
Every config cell × query, throughput in M tuples/s. The config label is the benchmark slug: `h<horizon>_pp<preemptive_predictor>_p<pool>_t<threads>_<predictor>`. Color is normalized per column (per query) so each query's best/worst spread is visible; the annotation is the raw value. Failed cells show as blank.

In [ ]:
df['config'] = (
    'h' + df['preemptive_horizon_ms'].astype('Int64').astype(str)
    + '_pp' + df['preemptive_predictor'].astype(str)
    + '_p' + df['recycle_pool_size'].astype('Int64').astype(str)
    + '_t' + df['threads'].astype('Int64').astype(str)
    + '_' + df['predictor'].astype(str)
)
pivot = df.pivot_table(index='config', columns='query', values='tps_M', sort=True)
norm = pivot.div(pivot.max(axis=0), axis=1)  # per-query (column) normalization
fig, ax = plt.subplots(figsize=(2.2 * len(QUERIES) + 3, 0.32 * len(pivot) + 1.5))
sns.heatmap(norm, annot=pivot, fmt='.2f', cmap='rocket', linewidths=0.4,
            annot_kws={'fontsize': 7}, cbar_kws={'label': 'frac of column max'}, ax=ax)
ax.set_xlabel('')
ax.set_ylabel('config')
ax.set_title('Throughput (M tuples/s) per config × query  (color = frac of per-query max)')
fig.tight_layout()
plt.show()

## 6. Failures
Cells that didn't produce a result, by reason. Empty table = clean sweep.

In [ ]:
fails = df[df['failed']][['query', 'corner', 'threads', 'preemptive_predictor', 'predictor', 'failure_reason']]
print(f'{len(fails)} failed / {len(df)} cells')
if len(fails):
    display(fails.reset_index(drop=True))
    print('\nby reason:')
    print(fails['failure_reason'].value_counts())